# Notebook 03 — User Threat Check
Geocodes a user-supplied address and intersects it against NHC hazard polygons.

**Docs:**
- Nominatim: https://nominatim.org/release-docs/develop/api/Search/
- geopandas intersect: https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.contains.html

In [ ]:
import sys
sys.path.insert(0, '..')

from geocoder import geocode_user_location
from data_fetcher import get_active_storms, parse_hurricane_gis, fetch_noaa_storm_surge
from gis_processor import is_within_threat_zone

print('Imports OK')

## 1. Geocode User Location
Change `USER_LOCATION` to any address, city, or postcode.

In [ ]:
USER_LOCATION = 'Tampa, FL'   # <-- change this

geo = geocode_user_location(USER_LOCATION)

if geo['success']:
    print(f'Resolved:    {geo["display_name"]}')
    print(f'Latitude:    {geo["lat"]}')
    print(f'Longitude:   {geo["lon"]}')
else:
    print(f'Could not geocode "{USER_LOCATION}". Try a more specific address.')

## 2. Fetch Active Storms & Evaluate Threat

In [ ]:
if not geo['success']:
    print('Geocoding failed — cannot check threat.')
else:
    lat, lon = geo['lat'], geo['lon']
    storms   = get_active_storms()

    if not storms:
        print('No active tropical cyclones — no threat assessment needed.')
    else:
        for storm in storms:
            print(f"\n--- {storm['storm_type']} {storm['name']} ({storm['storm_id']}) ---")
            gis    = parse_hurricane_gis(storm)
            surge  = fetch_noaa_storm_surge(storm)
            threat = is_within_threat_zone(lat, lon, gis, surge.get('surge_gdf'))

            print(f'  Threat level:   {threat.threat_level}')
            print(f'  In cone:        {threat.in_cone}')
            print(f'  In warning:     {threat.in_warning}')
            print(f'  In watch:       {threat.in_watch}')
            print(f'  In surge zone:  {threat.in_surge_zone}')
            if threat.distance_km:
                print(f'  Distance to track: {threat.distance_km:.1f} km')
            print(f'  Summary: {threat.threat_summary}')

## 3. Test Multiple Locations

In [ ]:
test_locations = [
    'Miami, FL',
    'Kingston, Jamaica',
    'Nassau, Bahamas',
    'New York, NY',
    'London, UK',
]

if not storms:
    print('No active storms to test against.')
else:
    storm = storms[0]  # Use first active storm
    gis   = parse_hurricane_gis(storm)
    surge = fetch_noaa_storm_surge(storm)

    print(f'Threat assessment for {storm["storm_type"]} {storm["name"]}:\n')
    print(f'{"Location":<25} {"Threat Level":<12} {"In Cone":<10} {"In Warning"}')
    print('-' * 60)

    for loc in test_locations:
        g = geocode_user_location(loc)
        if g['success']:
            t = is_within_threat_zone(g['lat'], g['lon'], gis, surge.get('surge_gdf'))
            print(f"{loc:<25} {t.threat_level:<12} {str(t.in_cone):<10} {t.in_warning}")
        else:
            print(f"{loc:<25} (geocoding failed)")